In [1]:
import sys
MODULEPATH="../../"
sys.path.append(MODULEPATH)
sys.path.append('../')
from connectivity.plot_results import plot_results_with_paths, plot_saddle_tree_with_function, plot_full_connectivity_tree_style

from critical_points.torch_critical_point_finder import find_critical_points_torch, save_critical_points_to_csv
from test_schwefel.schwefel_function_nnmodule import Schwefel3D, schwefel_loss  # your existing module

import matplotlib.pyplot as plt
import pandas as pd


In [2]:
model_builder = lambda p: Schwefel3D(*p)

In [3]:
# Set parameters
BOUNDS = {'x1': (0.0, 1.0), 'x2': (0.0, 1.0), 'x3': (0.0, 1.0)}
DIM = 3
ATTEMPTS = 1024
MIN_DISTANCE = 0.01

In [4]:
try:
    resume_df = pd.read_csv("critical_points_schwefel_3D.csv", index_col=0)
    resume_df = resume_df[resume_df['type']=='minimum']
    display(resume_df)
except FileNotFoundError:
    resume_df = None


,x1,x2,x3,f_value,type
0,0.341809,0.824375,0.824375,2.171397e-01,minimum
1,0.034551,0.034551,0.034551,1.066044e+00,minimum
2,0.034551,0.034551,0.824375,7.106958e-01,minimum
3,0.341809,0.341809,0.341809,6.514190e-01,minimum
4,0.034551,0.824375,0.824375,3.553479e-01,minimum
5,0.341809,0.034551,0.034551,9.278355e-01,minimum
6,0.034551,0.824375,0.034551,7.106958e-01,minimum
7,0.824375,0.341809,0.824375,2.171397e-01,minimum
8,0.341809,0.341809,0.034551,7.896273e-01,minimum
9,0.824375,0.341809,0.341809,4.342794e-01,minimum


In [5]:
minima, maxima, saddles = find_critical_points_torch(
    model_builder=model_builder,
    minima_only=True,
    loss_func=schwefel_loss,
    input = [],             # placeholder model input for compatibility (Schwefel model has no model input)
    bounds=BOUNDS,
    dimension=DIM,
    num_attempts=ATTEMPTS,
    min_distance=MIN_DISTANCE,
    device="cpu",
    lr=5e-2,
    max_iter=5_000,
    resume_df=resume_df,  # 👈 here!
    seed=1101
)



Loaded 27 valid points from dataframe.
Resuming with 27 loaded points.

__________________________________________________________________________________________
Attempt 1/1024:
Starting point: [0.25317813 0.88321528 0.5628975 ]
Arrival point: [0.34180722 0.8243742  0.82437393]
check grad: tensor([-1.1508e-04, -4.0400e-05, -5.4370e-05])
tensor(0.0001)
Converged to critical point with ||grad||=1.34e-04 < 1e-05
Point [0.34180722 0.8243742  0.82437393] is too close to existing minimum at [0.34180945 0.82437499 0.82437499] (distance 0.000003 < 0.01)
Point [0.34180722 0.8243742  0.82437393] is too close to existing minimum. Rejecting.

__________________________________________________________________________________________
Attempt 2/1024:
Starting point: [0.77770659 0.06539727 0.25324407]
Arrival point: [0.82437224 0.03455237 0.34180911]
check grad: tensor([-1.4061e-04,  8.3701e-05, -1.7317e-05])
tensor(0.0002)
Converged to critical point with ||grad||=1.65e-04 < 1e-05
Point [0.82437224

In [6]:
# Save to CSV
results_df = save_critical_points_to_csv(minima, maxima, saddles, dimension=DIM, filename="critical_points_schwefel_torch.csv")

results_df = pd.concat((resume_df,results_df), ignore_index=True)

display(results_df)

Saved critical_points_schwefel_torch.csv


,x1,x2,x3,f_value,type
0,0.341809,0.824375,0.824375,2.171397e-01,minimum
1,0.034551,0.034551,0.034551,1.066044e+00,minimum
2,0.034551,0.034551,0.824375,7.106958e-01,minimum
3,0.341809,0.341809,0.341809,6.514190e-01,minimum
4,0.034551,0.824375,0.824375,3.553479e-01,minimum
5,0.341809,0.034551,0.034551,9.278355e-01,minimum
6,0.034551,0.824375,0.034551,7.106958e-01,minimum
7,0.824375,0.341809,0.824375,2.171397e-01,minimum
8,0.341809,0.341809,0.034551,7.896273e-01,minimum
9,0.824375,0.341809,0.341809,4.342794e-01,minimum
